### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\vmoni\AppData\Local\Temp\ipykernel_3648\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Projects\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: AI.pdf
  ✓ Loaded 128 pages

Processing: DL.pdf
  ✓ Loaded 170 pages

Processing: ML.pdf
  ✓ Loaded 120 pages

Total documents loaded: 418


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2018-07-20T11:07:01+05:30', 'author': 'Rati Ranjan Dash', 'moddate': '2018-07-20T11:07:01+05:30', 'source': '..\\data\\pdf\\AI.pdf', 'total_pages': 128, 'page': 0, 'page_label': '1', 'source_file': 'AI.pdf', 'file_type': 'pdf'}, page_content='1 \n \nLECTURE NOTES \nON \nARTIFICIAL INTELLIGENCE \n \n \n \n \n \nPREPARED BY \nDR. PRASHANTA KUMAR PATRA \nCOLLEGE OF ENGINEERING AND TECHNOLOGY , \nBHUBANESWAR'),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2018-07-20T11:07:01+05:30', 'author': 'Rati Ranjan Dash', 'moddate': '2018-07-20T11:07:01+05:30', 'source': '..\\data\\pdf\\AI.pdf', 'total_pages': 128, 'page': 1, 'page_label': '2', 'source_file': 'AI.pdf', 'file_type': 'pdf'}, page_content='2 \n \n \nARTIFICIAL INTELLIGENCE SYLLABUS \nModule 1                                                                             

In [5]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [6]:
chunks=split_documents(all_pdf_documents)
chunks

Split 418 documents into 1019 chunks

Example chunk:
Content: 1 
 
LECTURE NOTES 
ON 
ARTIFICIAL INTELLIGENCE 
 
 
 
 
 
PREPARED BY 
DR. PRASHANTA KUMAR PATRA 
COLLEGE OF ENGINEERING AND TECHNOLOGY , 
BHUBANESWAR...
Metadata: {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2018-07-20T11:07:01+05:30', 'author': 'Rati Ranjan Dash', 'moddate': '2018-07-20T11:07:01+05:30', 'source': '..\\data\\pdf\\AI.pdf', 'total_pages': 128, 'page': 0, 'page_label': '1', 'source_file': 'AI.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2018-07-20T11:07:01+05:30', 'author': 'Rati Ranjan Dash', 'moddate': '2018-07-20T11:07:01+05:30', 'source': '..\\data\\pdf\\AI.pdf', 'total_pages': 128, 'page': 0, 'page_label': '1', 'source_file': 'AI.pdf', 'file_type': 'pdf'}, page_content='1 \n \nLECTURE NOTES \nON \nARTIFICIAL INTELLIGENCE \n \n \n \n \n \nPREPARED BY \nDR. PRASHANTA KUMAR PATRA \nCOLLEGE OF ENGINEERING AND TECHNOLOGY , \nBHUBANESWAR'),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2018-07-20T11:07:01+05:30', 'author': 'Rati Ranjan Dash', 'moddate': '2018-07-20T11:07:01+05:30', 'source': '..\\data\\pdf\\AI.pdf', 'total_pages': 128, 'page': 1, 'page_label': '2', 'source_file': 'AI.pdf', 'file_type': 'pdf'}, page_content='2 \n \n \nARTIFICIAL INTELLIGENCE SYLLABUS \nModule 1                                                                             

### embedding And vectorStoreDB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3138.92it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\vmoni\AppData\Local\Temp\ipykernel_3648\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [9]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 1019


In [10]:
chunks

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2018-07-20T11:07:01+05:30', 'author': 'Rati Ranjan Dash', 'moddate': '2018-07-20T11:07:01+05:30', 'source': '..\\data\\pdf\\AI.pdf', 'total_pages': 128, 'page': 0, 'page_label': '1', 'source_file': 'AI.pdf', 'file_type': 'pdf'}, page_content='1 \n \nLECTURE NOTES \nON \nARTIFICIAL INTELLIGENCE \n \n \n \n \n \nPREPARED BY \nDR. PRASHANTA KUMAR PATRA \nCOLLEGE OF ENGINEERING AND TECHNOLOGY , \nBHUBANESWAR'),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2018-07-20T11:07:01+05:30', 'author': 'Rati Ranjan Dash', 'moddate': '2018-07-20T11:07:01+05:30', 'source': '..\\data\\pdf\\AI.pdf', 'total_pages': 128, 'page': 1, 'page_label': '2', 'source_file': 'AI.pdf', 'file_type': 'pdf'}, page_content='2 \n \n \nARTIFICIAL INTELLIGENCE SYLLABUS \nModule 1                                                                             

In [11]:
### Convert the text to embeddings

texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 1019 texts...


Batches: 100%|██████████| 32/32 [00:39<00:00,  1.23s/it]


Generated embeddings with shape: (1019, 384)
Adding 1019 documents to vector store...
Successfully added 1019 documents to vector store
Total documents in collection: 2038


### Retriever Pipeline From VectorStore

In [12]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [13]:
rag_retriever

In [14]:
rag_retriever.retrieve("What is Machine Learning ?")

Retrieving documents for query: 'What is Machine Learning ?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 51.10it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_5729b421_690',
  'content': '1 \n \nUNIT I  \nIntroduction to Machine Learning \n1. Introduction \n \n1.1 What Is Machine Learning?  \nMachine learning is programming computers to optimize a performance criterion using example \ndata or past experience. We have a model defined up to some parameters, and learning is the \nexecution of a computer program to optimize the parameters of the model using the training data or \npast experience. The model may be predictive to make predictions in the future, or descriptive to gain \nknowledge from data, or both. \nArthur Samuel, an early American leader in the field of computer gaming and artificial intellige nce, \ncoined the term “Machine Learning” in 1959 while at IBM. He defined machine learning as “the field of \nstudy that gives computers the ability to learn without being explicitly programmed.” However, there is \nno universally accepted definition for machine learning. Different authors define the term differently. \n \nDef

In [15]:
rag_retriever.retrieve("Hill Climbing")

Retrieving documents for query: 'Hill Climbing'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 73.77it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_a94e43c2_115',
  'content': 'worse than than the current state. Since hill climbing uses greedy approach, it will not \nmove to the worse state and terminate itself. The process will end even though a better \nsolution may exist. \nTo overcome local maximum problem : Utilize backtracking technique. Maintain a list of \nvisited states. If the search reaches an undesirable state, it can backtrack to the previous \nconfiguration and explore a new path. \n2. Plateau : On plateau all neighbors have same value . Hence, it is not possible to select the \nbest direction. \nTo overcome plateaus : Make a big jump. Randomly select a state far away from current state. \nChances are that we will land at a non-plateau region \n3. Ridge : Any point on a ridge can look like peak because movement in all possible \ndirections is downward. Hence the algorithm stops when it reaches this state. \nTo overcome Ridge : In this kind of obstacle, use two or more rules before testing. It \nimplies m

### RAG Pipeline- VectorDB To LLM Output Generation

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

### Integration Vectordb Context Pipeline with LLM Output

In [ ]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):

    results = retriever.retrieve(query, top_k=top_k)

    context = "\n\n".join(
        [doc["content"] for doc in results]
    ) if results else ""

    if not context:
        return "No relevant context found."

    prompt = f"""
Use the following context to answer the question.

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.invoke(prompt)

    return response.content

In [22]:
answer=rag_simple("How does a Hidden Layer work?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'How does a Hidden Layer work?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 90.90it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


A Hidden Layer works by using layers of mathematical functions, each designed to produce an output specific to an intended result. These functions, such as squashing functions, take an input and produce an output value between 0 and 1, which is useful for defining probability. Hidden layers allow the neural network to break down the function into specific transformations of the data, with each layer specialized to produce a defined output. For example, a hidden layer that identifies human eyes and ears can be used in conjunction with subsequent layers to identify faces in images, and similarly, a hidden layer that identifies wheels can be used with additional layers to identify cars.


### Enhanced RAG Pipeline Features

In [24]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Reinforcement learning", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Reinforcement learning'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 35.59it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Reinforcement learning is a type of machine learning where an agent learns to take actions to maximize a reward signal from an environment, facing challenges such as delayed rewards and the need for exploration.
Sources: [{'source': 'ML.pdf', 'page': 90, 'score': 0.2636926770210266, 'preview': '86 \n \n \n \n \nReinforcement learning problem characteristics \n \n1. Delayed reward: The task of the agent is to learn a target function 𝜋 that maps from the current state \ns to the optimal action a = 𝜋 (s). In reinforcement learning, training information is not available in (s, 𝜋 \n(s)). Instead, the tr...'}, {'source': 'ML.pdf', 'page': 90, 'score': 0.2636926770210266, 'preview': '86 \n \n \n \n \nReinforcement learning problem characteristics \n \n1. Delayed reward: The task of the agent is to learn a target function 𝜋 that maps from the current state \ns to the optimal action a = 𝜋 (s). In reinforcement learning, training information is not available in (s, 𝜋 \n(s)). Instead, the

In [27]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is Hill Climbing ?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is Hill Climbing ?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 36.05it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
 A heuristic function is a function that will rank all the possible alternatives at any 
branching step in search algorithm based on the available information. It helps the 
algorithm to select the best route out of possible routes. 
Features of Hill

 Climbing 
1. Variant of generate and test algorithm : It is a variant of generate and test algorithm. The 
generate and test algorithm is as follows : 
 
1. Generate a possible solutions. 
2. Test to see if this is the expected solution. 
3. If the solution has been found quit else go to step 1. 
Hence we call Hill climbing as a variant of generate and test algorithm as it takes the feedback 
from test procedure. Then this feedback is utilized by the generator in deciding the next move in 
search space. 
2. Uses the Greedy approach : At any point in state space, the search moves in that direction 
only which optimizes the cost of function with the hope of finding the optimal solution at 
the end. 
Types of Hill Climbing

 A heuristic function is a function that will rank all the possible alternatives at any 
branching step in search algorithm based on the available information. It helps the 
algorithm to select the best route out of possible routes. 
Features of Hill Climbing 
1. Var

In [28]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is Cricket ?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is Cricket ?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 69.70it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)



Final Answer: No relevant context found.
Summary: There is no text to summarize as the provided answer states "No relevant context found." This indicates that there is no available information to work with, resulting in no summary being possible.
History: {'question': 'what is Cricket ?', 'answer': 'No relevant context found.', 'sources': [], 'summary': 'There is no text to summarize as the provided answer states "No relevant context found." This indicates that there is no available information to work with, resulting in no summary being possible.'}
